# 03 — Clustering / segmentación de perfiles

Tarea académica 3 (`prd.md` §5, §7.3). Segmentamos a los solicitantes en perfiles de riesgo mediante clustering no supervisado, sobre la tabla de features de la Tarea 2 (`data/processed/features.parquet`).

**Plan:**

1. Cargar `features.parquet` y quedarnos solo con la porción `IS_TRAIN` (356,255 filas totales, pero el perfilado por `TARGET` solo tiene sentido donde lo conocemos). El clustering en sí no usa `TARGET` como input, solo se usa después para caracterizar los segmentos.
2. Elegir un subconjunto acotado e interpretable de features para clustering (no las 304 columnas completas): usar todo el one-hot de categóricas distorsionaría la distancia euclídea (dominarían variables binarias dispersas sobre las continuas de negocio) y no aporta a un perfil interpretable. Priorizamos variables demográficas, ratios de negocio y resúmenes de historial crediticio por tabla relacional.
3. Imputar (mediana) y escalar (`StandardScaler`) el subconjunto elegido, requisito de K-Means por ser un método basado en distancia.
4. Elegir la cantidad de clusters `k` con método del codo (inertia) y silhouette score sobre un rango razonable.
5. Ajustar K-Means final, perfilar cada segmento (medias de las variables, tasa de default, tamaño) y visualizar con PCA 2D (Plotly).
6. Documentar hallazgos en `docs/informe-final.md` §3, con la misma lógica de justificar antes de mostrar resultados que usamos en Tareas 1 y 2.

In [1]:
import pandas as pd
import numpy as np

DATA_DIR = "../data/processed"

features = pd.read_parquet(f"{DATA_DIR}/features.parquet")
print(features.shape)
features["IS_TRAIN"].value_counts()

(356255, 304)


IS_TRAIN
1    307511
0     48744
Name: count, dtype: int64

## Selección de features para clustering

**Por qué un subconjunto acotado y no las 304 columnas:** K-Means mide distancia euclídea entre puntos. Si incluyéramos las 140 columnas de one-hot de la Tarea 2, esas variables binarias dispersas (la mayoría en 0) dominarían la distancia por pura cantidad, aplastando el aporte de las variables continuas de negocio (ingreso, ratios, utilización de tarjeta) que son las que en la Tarea 1/2 mostraron señal real. Además, un perfil de riesgo con 304 dimensiones no es interpretable para un risk manager (persona B del PRD, US-B1): el objetivo es un puñado de segmentos que se puedan describir en una frase.

Elegimos tres grupos de variables, todas ya validadas por su señal o su rol interpretativo en Tareas 1-2:
- **Demográficas/capacidad de pago:** edad, antigüedad laboral, ingreso, cantidad de hijos/miembros de familia.
- **Ratios de negocio:** `credit_to_income`, `annuity_to_income`, `credit_to_goods`.
- **Resumen de historial crediticio** (una variable representativa por tabla relacional, la de mayor señal encontrada en la Tarea 2): `bureau_active_count`, `bb_months_count_mean`, `prev_approval_rate`, `pos_dpd_rate`, `cc_utilization_mean`, `inst_late_rate`, más los 6 flags de "sin registro" (la ausencia de historial es en sí misma parte del perfil, no solo un dato faltante a rellenar).

In [2]:
cluster_df = features.loc[features["IS_TRAIN"] == 1].copy()

cluster_df["AGE_YEARS"] = -cluster_df["DAYS_BIRTH"] / 365
cluster_df["EMPLOYED_YEARS"] = -cluster_df["DAYS_EMPLOYED"] / 365

cluster_cols = [
    "AGE_YEARS", "EMPLOYED_YEARS", "AMT_INCOME_TOTAL", "CNT_CHILDREN", "CNT_FAM_MEMBERS",
    "credit_to_income", "annuity_to_income", "credit_to_goods",
    "bureau_active_count", "bb_months_count_mean", "prev_approval_rate",
    "pos_dpd_rate", "cc_utilization_mean", "inst_late_rate",
    "bureau_no_record", "bb_no_record", "prev_no_record",
    "pos_no_record", "cc_no_record", "inst_no_record",
]

print(f"SK_ID_CURR en cluster_df: {cluster_df['SK_ID_CURR'].is_unique}")
print(f"Filas: {len(cluster_df)}")
cluster_df[cluster_cols].describe()

SK_ID_CURR en cluster_df: True
Filas: 307511


,AGE_YEARS,EMPLOYED_YEARS,AMT_INCOME_TOTAL,CNT_CHILDREN,CNT_FAM_MEMBERS,credit_to_income,annuity_to_income,credit_to_goods,bureau_active_count,bb_months_count_mean,prev_approval_rate,pos_dpd_rate,cc_utilization_mean,inst_late_rate
count,307511.000000,252137.000000,3.075110e+05,307511.000000,307509.000000,307511.000000,307499.000000,307233.000000,263491.000000,92231.000000,291057.000000,289444.000000,86036.000000,291643.000000
mean,43.936973,6.531971,1.687979e+05,0.417052,2.152665,3.957570,0.180930,1.122995,2.056689,27.477237,0.748855,0.021641,0.326981,0.029876
std,11.956133,6.406466,2.371231e+05,0.722121,0.910682,2.689728,0.094574,0.124045,1.787834,16.827669,0.262332,0.079927,0.325396,0.069434
min,20.517808,-0.000000,2.565000e+04,0.000000,1.000000,0.004808,0.000224,0.150000,0.000000,1.000000,0.000000,0.000000,-0.084848,0.000000
25%,34.008219,2.101370,1.125000e+05,0.000000,2.000000,2.018667,0.114782,1.000000,1.000000,15.000000,0.500000,0.000000,0.000000,0.000000
50%,43.150685,4.515068,1.471500e+05,0.000000,2.000000,3.265067,0.162833,1.118800,2.000000,24.750000,0.800000,0.000000,0.253387,0.000000
75%,53.923288,8.698630,2.025000e+05,1.000000,3.000000,5.159880,0.229067,1.198000,3.000000,37.000000,1.000000,0.000000,0.592000,0.026316
max,69.120548,49.073973,1.170000e+08,19.000000,20.000000,84.736842,1.875965,6.000000,32.000000,97.000000,1.000000,1.000000,2.138790,1.000000


## Imputación y escalado

Las variables de historial relacional tienen NaN por la misma razón que en la Tarea 2 (sin registro en esa tabla): `bb_months_count_mean` y `cc_utilization_mean` son las de mayor faltante (~70-77% sobre esta selección de columnas, algo mayor al total de `IS_TRAIN` porque acá ya filtramos). Como el flag `*_no_record` correspondiente ya viaja como columna aparte, imputar con la mediana acá no pierde esa señal, solo evita que K-Means falle por NaN.

Usamos mediana (no media) por robustez a outliers, dado que varias columnas (`AMT_INCOME_TOTAL`, `credit_to_income`) tienen colas largas visibles en el `describe()` anterior. Escalamos con `StandardScaler` porque K-Means es sensible a la escala: sin estandarizar, `AMT_INCOME_TOTAL` (rango de cientos de miles) dominaría por completo sobre `bureau_active_count` (rango de unidades).

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_imputed = imputer.fit_transform(cluster_df[cluster_cols])
X_scaled = scaler.fit_transform(X_imputed)

print(X_scaled.shape)
print(f"NaN restantes: {np.isnan(X_scaled).sum()}")

(307511, 20)
NaN restantes: 0


## Elección de `k`: método del codo + silhouette

Probamos `k` entre 2 y 8. La inertia (suma de distancias al cuadrado dentro de cada cluster) se calcula sobre el dataset completo, es barata. El silhouette score es costoso (compara cada punto contra todos los demás, O(n²)) para 307,511 filas, así que lo estimamos sobre una muestra aleatoria de 10,000 puntos (parámetro `sample_size` de `silhouette_score`) con semilla fija para que sea reproducible.

In [4]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

k_range = range(2, 9)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels, sample_size=10_000, random_state=42)
    silhouettes.append(sil)
    print(f"k={k}: inertia={km.inertia_:.0f}, silhouette={sil:.4f}")

k_selection = pd.DataFrame({"k": list(k_range), "inertia": inertias, "silhouette": silhouettes})
k_selection

k=2: inertia=5304566, silhouette=0.4515
k=3: inertia=4848130, silhouette=0.1345
k=4: inertia=4558916, silhouette=0.1377
k=5: inertia=4307398, silhouette=0.1380
k=6: inertia=4051395, silhouette=0.1483
k=7: inertia=3838396, silhouette=0.1367
k=8: inertia=3692234, silhouette=0.1405


,k,inertia,silhouette
0,2,5.304566e+06,0.451496
1,3,4.848130e+06,0.134457
2,4,4.558916e+06,0.137701
3,5,4.307398e+06,0.137952
4,6,4.051395e+06,0.148281
5,7,3.838396e+06,0.136702
6,8,3.692234e+06,0.140455


**Lectura de la tabla anterior:** el silhouette cae fuerte de `k=2` (0.45) a `k=3+` (~0.13-0.15), y la inertia decrece de forma suave y constante, sin un codo marcado. Dos hipótesis a chequear antes de elegir `k` a ciegas por el silhouette más alto:

1. El salto en `k=2` puede ser un split trivial dominado por alguna de las 6 variables binarias dispersas (los flags `*_no_record`, con hasta 71% de un solo valor), no un perfil de riesgo real. Lo chequeamos perfilando ese `k=2` a continuación.
2. Un silhouette bajo pero estable en `k=3..8` es esperable en datos de crédito: a diferencia de datos sintéticos con clusters bien separados, los perfiles de clientes suelen ser un continuo (edad, ingreso, uso de crédito), no grupos discretos con fronteras nítidas.

In [5]:
km2 = KMeans(n_clusters=2, random_state=42, n_init=10)
labels2 = km2.fit_predict(X_scaled)

profile2 = cluster_df[cluster_cols].assign(cluster=labels2).groupby("cluster").mean()
sizes2 = pd.Series(labels2).value_counts().sort_index()

print(sizes2)
profile2.T

0     16149
1    291362
Name: count, dtype: int64


cluster,0,1
AGE_YEARS,43.047736,43.986260
EMPLOYED_YEARS,5.812134,6.572680
AMT_INCOME_TOTAL,207562.585397,166649.352933
CNT_CHILDREN,0.313704,0.422780
CNT_FAM_MEMBERS,2.013996,2.160350
credit_to_income,4.307762,3.938161
annuity_to_income,0.168307,0.181630
credit_to_goods,1.109524,1.123742
bureau_active_count,1.762134,2.072871
bb_months_count_mean,30.740454,27.347104


**Confirmado:** el `k=2` separa clientes sin historial previo con Home Credit (cluster 0: 96-99% en `prev_no_record`/`pos_no_record`/`inst_no_record`) de clientes con historial previo (cluster 1). Es un split real pero demasiado grueso -"nuevo" vs. "recurrente"- y no el perfil de riesgo multidimensional que buscamos para US-B1.

## Decisión final de `k`

Entre `k=3..8` las diferencias de silhouette (0.134-0.148) son chicas, probablemente dentro del ruido de estimarlo sobre una muestra de 10,000 puntos, y la inertia no muestra un codo marcado. Elegimos `k=5`: un número de segmentos manejable para uso de negocio (persona B, risk manager, US-B1), con silhouette (0.138) prácticamente igual al máximo observado (`k=6`, 0.148).

In [6]:
K_FINAL = 5

kmeans_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
cluster_df["cluster"] = kmeans_final.fit_predict(X_scaled)

cluster_sizes = cluster_df["cluster"].value_counts().sort_index()
cluster_default_rate = cluster_df.groupby("cluster")["TARGET"].mean()

pd.DataFrame({"size": cluster_sizes, "default_rate": cluster_default_rate})

,size,default_rate
cluster,,
0,13862,0.128625
1,71956,0.086372
2,16091,0.058169
3,168980,0.072908
4,36622,0.097510


In [7]:
pd.set_option("display.max_columns", None)
profile_final = cluster_df.groupby("cluster")[cluster_cols].mean()
profile_final["default_rate"] = cluster_default_rate
profile_final["size"] = cluster_sizes
profile_final.T

cluster,0,1,2,3,4
AGE_YEARS,43.988418,36.789527,43.035590,47.131864,43.615309
EMPLOYED_YEARS,6.563615,5.920909,5.808953,7.186373,5.469864
AMT_INCOME_TOTAL,159290.572315,173399.019883,207618.332457,168075.313396,149633.502471
CNT_CHILDREN,0.374188,1.463853,0.312908,0.027926,0.217738
CNT_FAM_MEMBERS,2.097966,3.382581,2.013301,1.702349,1.895855
credit_to_income,3.958745,3.860071,4.308735,3.954136,4.010246
annuity_to_income,0.187351,0.179899,0.168241,0.178907,0.195434
credit_to_goods,1.131214,1.126570,1.109445,1.121265,1.126800
bureau_active_count,2.001716,2.095168,1.761703,2.068914,NaN
bb_months_count_mean,29.557100,26.245177,30.711779,27.650199,NaN


## Visualización: PCA 2D

Reducimos las 20 dimensiones a 2 componentes principales solo para graficar (no para el clustering en sí, que ya corrió sobre el espacio completo). Tomamos una muestra de 15,000 puntos para que el scatter sea legible y liviano.

In [9]:
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "iframe"

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plot_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
plot_df["cluster"] = cluster_df["cluster"].values
plot_sample = plot_df.sample(15_000, random_state=42)

print(f"Varianza explicada por PC1/PC2: {pca.explained_variance_ratio_.sum():.3f}")

fig = px.scatter(
    plot_sample, x="PC1", y="PC2", color=plot_sample["cluster"].astype(str),
    title="Segmentos de clientes (K-Means, k=5) proyectados en PCA 2D",
    labels={"color": "cluster"}, opacity=0.5,
)
fig.show()

Varianza explicada por PC1/PC2: 0.247


## Persistencia: segmentos sobre toda la población

El clustering se ajustó (`fit`) solo sobre train, siguiendo la buena práctica de no ajustar transformaciones con datos de test. Para que el segmento quede disponible como feature de negocio también sobre los solicitantes de test (US-B1 no distingue train/test, es una propiedad de cualquier cliente), aplicamos (`transform`, no `fit`) el mismo `imputer`/`scaler`/`kmeans_final` ya ajustados a la población completa.

In [10]:
import os

all_df = features.copy()
all_df["AGE_YEARS"] = -all_df["DAYS_BIRTH"] / 365
all_df["EMPLOYED_YEARS"] = -all_df["DAYS_EMPLOYED"] / 365

X_all_imputed = imputer.transform(all_df[cluster_cols])
X_all_scaled = scaler.transform(X_all_imputed)
all_df["cluster"] = kmeans_final.predict(X_all_scaled)

segments = all_df[["SK_ID_CURR", "IS_TRAIN", "cluster"]].copy()
assert segments["SK_ID_CURR"].is_unique
print(segments.shape)
print(segments.groupby("IS_TRAIN")["cluster"].value_counts(normalize=True).sort_index())

OUT_DIR = "../data/processed"
os.makedirs(OUT_DIR, exist_ok=True)
segments.to_parquet(f"{OUT_DIR}/segments.parquet", index=False)
print(f"Guardado en {OUT_DIR}/segments.parquet: {segments.shape}")

(356255, 3)
IS_TRAIN  cluster
0         0          0.033050
          1          0.233998
          2          0.017746
          3          0.600915
          4          0.114291
1         0          0.045078
          1          0.233995
          2          0.052327
          3          0.549509
          4          0.119092
Name: proportion, dtype: float64
Guardado en ../data/processed/segments.parquet: (356255, 3)
